In [1]:
import io
import re
from pathlib import Path
import pandas as pd

base_dir = Path(".") 
all_xls = {}

for season_dir in [base_dir / "24-25", base_dir / "25-26"]:
    if not season_dir.exists():
        print(f"⚠️ Could not find folder: {season_dir.absolute()}")
        continue

    for file in sorted(season_dir.glob("*.xls")):
        name = f"{season_dir.name}_{file.stem}" 
        
        try:
            with open(file, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            tables = pd.read_html(io.StringIO(html_content))
            df = tables[0]
            all_xls[name] = df
            
        except ValueError:
            print(f"No tables found in {name}. Is the file completely empty?")
        except Exception as e:
            print(f"Error loading {name}: {e}")

# simplify naming
for name, df in all_xls.items():
    season_part, _, file_part = name.partition("_")
    season = season_part[:2]
    file_part = re.sub(r"^\d+\s*", "", file_part)
    file_part = file_part.replace("-", "_").replace(" ", "_")
    clean_name = f"{file_part}_{season}"
    clean_name = re.sub(r"[^0-9a-zA-Z_]+", "_", clean_name).strip("_")
    globals()[clean_name] = df
    print(f"Created df: {clean_name}")

Created df: scores_fixtures_24
Created df: shooting_for_24
Created df: shooting_against_24
Created df: keeping_for_24
Created df: keeping_against_24
Created df: misc_for_24
Created df: misc_against_24
Created df: scores_fixtures_25
Created df: shooting_for_25
Created df: shooting_against_25
Created df: keeping_for_25
Created df: keeping_against_25
Created df: misc_for_25
Created df: misc_against_25


In [2]:
scores_fixtures_24 = all_xls["24-25_1 scores-fixtures"]
shooting_for_24 = all_xls["24-25_2 shooting-for"]
shooting_against_24 = all_xls["24-25_3 shooting-against"]
keeping_for_24 = all_xls["24-25_4 keeping-for"]
keeping_against_24 = all_xls["24-25_5 keeping-against"]
misc_for_24 = all_xls["24-25_6 misc-for"]
misc_against_24 = all_xls["24-25_7 misc-against"]

scores_fixtures_25 = all_xls["25-26_1 scores-fixtures"]
shooting_for_25 = all_xls["25-26_2 shooting-for"]
shooting_against_25 = all_xls["25-26_3 shooting-against"]
keeping_for_25 = all_xls["25-26_4 keeping-for"]
keeping_against_25 = all_xls["25-26_5 keeping-against"]
misc_for_25 = all_xls["25-26_6 misc-for"]
misc_against_25 = all_xls["25-26_7 misc-against"]

In [20]:
print(scores_fixtures_24.columns)

Index(['Date', 'Time', 'Round', 'Day', 'Venue', 'Result', 'GF', 'GA',
       'Opponent', 'Poss', 'Attendance', 'Captain', 'Formation',
       'Opp Formation', 'Referee', 'Match Report', 'Notes'],
      dtype='str')


In [21]:
print(shooting_for_24.columns)

MultiIndex([(      'For Liverpool',         'Date'),
            (      'For Liverpool',         'Time'),
            (      'For Liverpool',        'Round'),
            (      'For Liverpool',          'Day'),
            (      'For Liverpool',        'Venue'),
            (      'For Liverpool',       'Result'),
            (      'For Liverpool',           'GF'),
            (      'For Liverpool',           'GA'),
            (      'For Liverpool',     'Opponent'),
            (           'Standard',          'Gls'),
            (           'Standard',           'Sh'),
            (           'Standard',          'SoT'),
            (           'Standard',         'SoT%'),
            (           'Standard',         'G/Sh'),
            (           'Standard',        'G/SoT'),
            (           'Standard',           'PK'),
            (           'Standard',        'PKatt'),
            ('Unnamed: 17_level_0', 'Match Report')],
           )


In [22]:
print(keeping_for_24.columns)

MultiIndex([(      'For Liverpool',         'Date'),
            (      'For Liverpool',         'Time'),
            (      'For Liverpool',        'Round'),
            (      'For Liverpool',          'Day'),
            (      'For Liverpool',        'Venue'),
            (      'For Liverpool',       'Result'),
            (      'For Liverpool',           'GF'),
            (      'For Liverpool',           'GA'),
            (      'For Liverpool',     'Opponent'),
            (        'Performance',         'SoTA'),
            (        'Performance',           'GA'),
            (        'Performance',        'Saves'),
            (        'Performance',        'Save%'),
            (        'Performance',           'CS'),
            (      'Penalty Kicks',        'PKatt'),
            (      'Penalty Kicks',          'PKA'),
            (      'Penalty Kicks',         'PKsv'),
            (      'Penalty Kicks',          'PKm'),
            ('Unnamed: 18_level_0', 'Match Rep

In [23]:
print(misc_for_24.columns)

MultiIndex([(      'For Liverpool',         'Date'),
            (      'For Liverpool',         'Time'),
            (      'For Liverpool',        'Round'),
            (      'For Liverpool',          'Day'),
            (      'For Liverpool',        'Venue'),
            (      'For Liverpool',       'Result'),
            (      'For Liverpool',           'GF'),
            (      'For Liverpool',           'GA'),
            (      'For Liverpool',     'Opponent'),
            (        'Performance',         'CrdY'),
            (        'Performance',         'CrdR'),
            (        'Performance',        '2CrdY'),
            (        'Performance',          'Fls'),
            (        'Performance',          'Fld'),
            (        'Performance',          'Off'),
            (        'Performance',          'Crs'),
            (        'Performance',          'Int'),
            (        'Performance',         'TklW'),
            (        'Performance',        'PK

In [6]:
# Independently reload each season's source files and verify the notebook dataframes.
from pathlib import Path
import io
import pandas as pd

source_tables = {}
for season_folder in (Path("24-25"), Path("25-26")):
    for source_file in sorted(season_folder.glob("*.xls")):
        with source_file.open("r", encoding="utf-8") as source:
            source_tables[f"{season_folder.name}_{source_file.stem}"] = pd.read_html(io.StringIO(source.read()))[0]

for table_key, source_df in source_tables.items():
    notebook_df = all_xls[table_key]
    print(f"{table_key}: rows={len(source_df)}, matches_notebook={source_df.equals(notebook_df)}")

for table_key in ("24-25_2 shooting-for", "24-25_3 shooting-against", "25-26_2 shooting-for", "25-26_3 shooting-against"):
    print(f"\\nLast rows of {table_key}:")
    print(source_tables[table_key].iloc[-2:, [0, 2, 5, 6, 7, 8, 9, 10, 11]].to_string(index=False))

24-25_1 scores-fixtures: rows=38, matches_notebook=True
24-25_2 shooting-for: rows=39, matches_notebook=True
24-25_3 shooting-against: rows=39, matches_notebook=True
24-25_4 keeping-for: rows=39, matches_notebook=True
24-25_5 keeping-against: rows=39, matches_notebook=True
24-25_6 misc-for: rows=39, matches_notebook=True
24-25_7 misc-against: rows=39, matches_notebook=True
25-26_1 scores-fixtures: rows=38, matches_notebook=True
25-26_2 shooting-for: rows=39, matches_notebook=True
25-26_3 shooting-against: rows=39, matches_notebook=True
25-26_4 keeping-for: rows=39, matches_notebook=True
25-26_5 keeping-against: rows=39, matches_notebook=True
25-26_6 misc-for: rows=39, matches_notebook=True
25-26_7 misc-against: rows=39, matches_notebook=True
\nLast rows of 24-25_2 shooting-for:
For Liverpool                                          Standard        
         Date        Round Result GF GA       Opponent      Gls  Sh SoT
   2025-05-25 Matchweek 38      D  1  1 Crystal Palace        1  14

In [7]:
def find_column(table, name):
    for column in table.columns:
        if (column[-1] if isinstance(column, tuple) else column) == name:
            return column
    raise KeyError(f"Column not found: {name}")


def match_rows(table):
    date_column = find_column(table, "Date")
    return table.loc[table[date_column].notna()].copy()


def numeric_total(table, column_name):
    rows = match_rows(table)
    column = find_column(rows, column_name)
    return int(pd.to_numeric(rows[column], errors="coerce").sum())


def season_metrics(fixtures, shooting_for, shooting_against):
    results = fixtures["Result"].value_counts()
    matches = len(fixtures)
    shots = numeric_total(shooting_for, "Sh")
    shots_on_target = numeric_total(shooting_for, "SoT")
    shots_allowed = numeric_total(shooting_against, "Sh")
    shots_on_target_allowed = numeric_total(shooting_against, "SoT")
    shooting_goals = numeric_total(shooting_for, "Gls")
    return {
        "matches": matches,
        "wins": int(results.get("W", 0)),
        "draws": int(results.get("D", 0)),
        "losses": int(results.get("L", 0)),
        "points": int(results.get("W", 0) * 3 + results.get("D", 0)),
        "goals_for": int(pd.to_numeric(fixtures["GF"]).sum()),
        "goals_against": int(pd.to_numeric(fixtures["GA"]).sum()),
        "clean_sheets": int((pd.to_numeric(fixtures["GA"]) == 0).sum()),
        "average_possession": float(pd.to_numeric(fixtures["Poss"]).mean()),
        "shots": shots,
        "shots_on_target": shots_on_target,
        "shooting_goals": shooting_goals,
        "shots_allowed": shots_allowed,
        "shots_on_target_allowed": shots_on_target_allowed,
        "points_per_match": (int(results.get("W", 0) * 3 + results.get("D", 0)) / matches),
        "shots_per_match": shots / matches,
        "shots_on_target_per_match": shots_on_target / matches,
        "shots_on_target_rate": shots_on_target / shots,
        "shot_conversion_rate": shooting_goals / shots,
        "shots_allowed_per_match": shots_allowed / matches,
        "shots_on_target_allowed_per_match": shots_on_target_allowed / matches,
    }


fixture_tables = {"24-25": scores_fixtures_24, "25-26": scores_fixtures_25}
notebook_metrics = {}
workbook_metrics = {}
for season_label, fixtures in fixture_tables.items():
    short_season = season_label[:2]
    notebook_metrics[season_label] = season_metrics(
        fixtures,
        all_xls[f"{season_label}_2 shooting-for"],
        all_xls[f"{season_label}_3 shooting-against"],
    )
    workbook_metrics[season_label] = season_metrics(
        source_tables[f"{season_label}_1 scores-fixtures"],
        source_tables[f"{season_label}_2 shooting-for"],
        source_tables[f"{season_label}_3 shooting-against"],
    )
    assert len(match_rows(all_xls[f"{season_label}_2 shooting-for"])) == len(fixtures)
    assert notebook_metrics[season_label] == workbook_metrics[season_label]

comparison = pd.DataFrame(notebook_metrics).T
print("Liverpool Premier League comparison: 38 matches per season")
print("Notebook dataframe metrics equal independently reloaded workbook metrics for both seasons.")
print(comparison.round(3).to_string())

Liverpool Premier League comparison: 38 matches per season
Notebook dataframe metrics equal independently reloaded workbook metrics for both seasons.
       matches  wins  draws  losses  points  goals_for  goals_against  clean_sheets  average_possession  shots  shots_on_target  shooting_goals  shots_allowed  shots_on_target_allowed  points_per_match  shots_per_match  shots_on_target_per_match  shots_on_target_rate  shot_conversion_rate  shots_allowed_per_match  shots_on_target_allowed_per_match
24-25     38.0  25.0    9.0     4.0    84.0       86.0           41.0          14.0              58.079  647.0            231.0            85.0          387.0                    141.0             2.211           17.026                      6.079                 0.357                 0.131                   10.184                              3.711
25-26     38.0  17.0    9.0    12.0    60.0       63.0           53.0          10.0              59.316  588.0            175.0            61.0       

In [30]:
misc_fields = ["CrdY", "CrdR", "2CrdY", "Fls", "Fld", "Off", "Crs", "Int", "TklW", "OG"]
zero_imputed_fields = {"Off"}


def paired_misc_frame(misc_for, misc_against, fixtures):
    for_rows = match_rows(misc_for)
    against_rows = match_rows(misc_against)
    assert len(for_rows) == len(against_rows) == len(fixtures) == 38

    def match_keys(rows):
        dates = pd.to_datetime(rows[find_column(rows, "Date")]).dt.strftime("%Y-%m-%d")
        opponents = rows[find_column(rows, "Opponent")].astype(str).str.strip().str.casefold()
        return list(zip(dates, opponents))

    for_keys = match_keys(for_rows)
    against_keys = match_keys(against_rows)
    assert for_keys == against_keys, "for/against rows are not aligned by fixture"

    fixture_dates = pd.to_datetime(fixtures["Date"]).dt.strftime("%Y-%m-%d")
    fixture_opponents = fixtures["Opponent"].astype(str).str.strip().str.casefold()
    fixture_keys = list(zip(fixture_dates, fixture_opponents))
    assert for_keys == fixture_keys, "misc rows do not match fixture dates and opponents"

    paired = pd.DataFrame({
        "date": fixture_dates.tolist(),
        "opponent": fixtures["Opponent"].tolist(),
        "result": fixtures["Result"].tolist(),
        "points": fixtures["Result"].map({"W": 3, "D": 1, "L": 0}).tolist(),
    })
    imputation_audit = {}
    for field in misc_fields:
        for_column = find_column(for_rows, field)
        against_column = find_column(against_rows, field)
        liverpool_values = pd.to_numeric(for_rows[for_column], errors="coerce")
        opponent_values = pd.to_numeric(against_rows[against_column], errors="coerce")
        if field in zero_imputed_fields:
            imputation_audit[f"Liverpool {field}"] = int(liverpool_values.isna().sum())
            imputation_audit[f"Opponents {field}"] = int(opponent_values.isna().sum())
            liverpool_values = liverpool_values.fillna(0)
            opponent_values = opponent_values.fillna(0)
        paired[f"liverpool_{field}"] = liverpool_values.to_numpy()
        paired[f"opponents_{field}"] = opponent_values.to_numpy()
    paired.attrs["imputation_audit"] = imputation_audit
    return paired


misc_inputs = {
    "24-25": (misc_for_24, misc_against_24, scores_fixtures_24),
    "25-26": (misc_for_25, misc_against_25, scores_fixtures_25),
}
paired_misc = {}
imputation_audit = {}
for season_label, (misc_for, misc_against, fixtures) in misc_inputs.items():
    notebook_pair = paired_misc_frame(misc_for, misc_against, fixtures)
    workbook_pair = paired_misc_frame(
        source_tables[f"{season_label}_6 misc-for"],
        source_tables[f"{season_label}_7 misc-against"],
        source_tables[f"{season_label}_1 scores-fixtures"],
    )
    assert notebook_pair.equals(workbook_pair), f"dataframe and workbook misc stats differ for {season_label}"
    assert notebook_pair.attrs["imputation_audit"] == workbook_pair.attrs["imputation_audit"]
    paired_misc[season_label] = notebook_pair
    imputation_audit[season_label] = notebook_pair.attrs["imputation_audit"]

print("Paired misc-stat analysis: all 38 fixtures align by date/opponent; dataframe and workbook calculations match.")
print("\\nMissing offsides replaced with zero (Liverpool / opponents):")
for season_label, audit in imputation_audit.items():
    print(season_label, audit)

summary_rows = []
for season_label, paired in paired_misc.items():
    for field in misc_fields:
        liverpool_values = paired[f"liverpool_{field}"]
        opponent_values = paired[f"opponents_{field}"]
        summary_rows.append({
            "season": season_label,
            "stat": field,
            "Liverpool_total": liverpool_values.sum(min_count=1),
            "Liverpool_per_match": liverpool_values.mean(),
            "Liverpool_reported_matches": int(liverpool_values.notna().sum()),
            "Opponent_total": opponent_values.sum(min_count=1),
            "Opponent_per_match": opponent_values.mean(),
            "Opponent_reported_matches": int(opponent_values.notna().sum()),
        })

misc_summary = pd.DataFrame(summary_rows)
print("\\nSeason totals and per-match rates (Liverpool perspective):")
print(misc_summary.round(2).to_string(index=False))

outcome_rows = []
for season_label, paired in paired_misc.items():
    for outcome_group, outcome_matches in (("Wins", paired[paired["points"] == 3]), ("Dropped points", paired[paired["points"] < 3])):
        row = {"season": season_label, "outcome": outcome_group, "matches": len(outcome_matches)}
        for field in ("Fls", "Fld", "CrdY", "Off", "Crs", "Int", "TklW", "OG"):
            row[f"Liverpool {field}/match"] = outcome_matches[f"liverpool_{field}"].mean()
            row[f"Opponents {field}/match"] = outcome_matches[f"opponents_{field}"].mean()
        outcome_rows.append(row)

outcome_summary = pd.DataFrame(outcome_rows)
print("\\nPer-match misc activity by result group (descriptive, not causal):")
print(outcome_summary.round(2).to_string(index=False))

Paired misc-stat analysis: all 38 fixtures align by date/opponent; dataframe and workbook calculations match.
\nMissing offsides replaced with zero (Liverpool / opponents):
24-25 {'Liverpool Off': 0, 'Opponents Off': 1}
25-26 {'Liverpool Off': 0, 'Opponents Off': 1}
\nSeason totals and per-match rates (Liverpool perspective):
season  stat  Liverpool_total  Liverpool_per_match  Liverpool_reported_matches  Opponent_total  Opponent_per_match  Opponent_reported_matches
 24-25  CrdY               67                 1.76                          38            84.0                2.21                         38
 24-25  CrdR                3                 0.08                          38             2.0                0.05                         38
 24-25 2CrdY                1                 0.03                          38             2.0                0.05                         38
 24-25   Fls              430                11.32                          38           371.0          

In [31]:
penalty_inputs = {
    "24-25": (shooting_for_24, shooting_against_24, keeping_for_24, keeping_against_24, scores_fixtures_24),
    "25-26": (shooting_for_25, shooting_against_25, keeping_for_25, keeping_against_25, scores_fixtures_25),
}


def penalty_match_values(table, fields):
    rows = match_rows(table)
    dates = pd.to_datetime(rows[find_column(rows, "Date")]).dt.strftime("%Y-%m-%d")
    opponents = rows[find_column(rows, "Opponent")].astype(str).str.strip().str.casefold()
    values = {
        field: pd.to_numeric(rows[find_column(rows, field)], errors="coerce").fillna(0).astype(int).to_numpy()
        for field in fields
    }
    missing = {
        field: int(pd.to_numeric(rows[find_column(rows, field)], errors="coerce").isna().sum())
        for field in fields
    }
    return list(zip(dates, opponents)), values, missing


def penalty_season_metrics(shoot_for, shoot_against, keep_for, keep_against, fixtures):
    fixture_dates = pd.to_datetime(fixtures["Date"]).dt.strftime("%Y-%m-%d")
    fixture_opponents = fixtures["Opponent"].astype(str).str.strip().str.casefold()
    fixture_keys = list(zip(fixture_dates, fixture_opponents))
    sf_keys, sf, sf_missing = penalty_match_values(shoot_for, ("PK", "PKatt"))
    sa_keys, sa, sa_missing = penalty_match_values(shoot_against, ("PK", "PKatt"))
    kf_keys, kf, kf_missing = penalty_match_values(keep_for, ("PKatt", "PKA", "PKsv", "PKm"))
    ka_keys, ka, ka_missing = penalty_match_values(keep_against, ("PKatt", "PKA", "PKsv", "PKm"))
    assert sf_keys == sa_keys == kf_keys == ka_keys == fixture_keys

    # Shooting records are the attacker's view; the opposite keeper table records the same attempts and goals allowed.
    assert (sf["PKatt"] == ka["PKatt"]).all()
    assert (sf["PK"] == ka["PKA"]).all()
    assert (sa["PKatt"] == kf["PKatt"]).all()
    assert (sa["PK"] == kf["PKA"]).all()
    assert (kf["PKatt"] == kf["PKA"] + kf["PKsv"] + kf["PKm"]).all()
    assert (ka["PKatt"] == ka["PKA"] + ka["PKsv"] + ka["PKm"]).all()

    liverpool_attempts = int(sf["PKatt"].sum())
    opponent_attempts = int(sa["PKatt"].sum())
    return {
        "Liverpool PK scored": int(sf["PK"].sum()),
        "Liverpool PK attempts": liverpool_attempts,
        "Liverpool PK conversion": sf["PK"].sum() / liverpool_attempts if liverpool_attempts else 0,
        "Opponent PK scored": int(sa["PK"].sum()),
        "Opponent PK attempts": opponent_attempts,
        "Opponent PK conversion": sa["PK"].sum() / opponent_attempts if opponent_attempts else 0,
        "Liverpool keeper PK faced": int(kf["PKatt"].sum()),
        "Liverpool keeper PK conceded": int(kf["PKA"].sum()),
        "Liverpool keeper PK saved": int(kf["PKsv"].sum()),
        "Liverpool keeper PK missed": int(kf["PKm"].sum()),
        "Opponent keeper PK faced": int(ka["PKatt"].sum()),
        "Opponent keeper PK conceded": int(ka["PKA"].sum()),
        "Opponent keeper PK saved": int(ka["PKsv"].sum()),
        "Opponent keeper PK missed": int(ka["PKm"].sum()),
        "source_missing_values": {"shoot_for": sf_missing, "shoot_against": sa_missing, "keeping_for": kf_missing, "keeping_against": ka_missing},
    }


penalty_metrics = {}
for season_label, (shoot_for, shoot_against, keep_for, keep_against, fixtures) in penalty_inputs.items():
    dataframe_metrics = penalty_season_metrics(shoot_for, shoot_against, keep_for, keep_against, fixtures)
    workbook_metrics = penalty_season_metrics(
        source_tables[f"{season_label}_2 shooting-for"],
        source_tables[f"{season_label}_3 shooting-against"],
        source_tables[f"{season_label}_4 keeping-for"],
        source_tables[f"{season_label}_5 keeping-against"],
        source_tables[f"{season_label}_1 scores-fixtures"],
    )
    assert dataframe_metrics == workbook_metrics, f"dataframe and workbook penalty metrics differ for {season_label}"
    penalty_metrics[season_label] = dataframe_metrics

penalty_comparison = pd.DataFrame(penalty_metrics).T.drop(columns="source_missing_values")
print("Penalty stats cross-check: shooting PK/PKatt equals opposite keeper PKA/PKatt, row by row.")
print("Dataframe and independently reloaded workbook metrics match.")
print(penalty_comparison.to_string())
print("\\nBlank source penalty values by dataframe and field:")
for season_label, metrics in penalty_metrics.items():
    print(season_label, metrics["source_missing_values"])

Penalty stats cross-check: shooting PK/PKatt equals opposite keeper PKA/PKatt, row by row.
Dataframe and independently reloaded workbook metrics match.
      Liverpool PK scored Liverpool PK attempts Liverpool PK conversion Opponent PK scored Opponent PK attempts Opponent PK conversion Liverpool keeper PK faced Liverpool keeper PK conceded Liverpool keeper PK saved Liverpool keeper PK missed Opponent keeper PK faced Opponent keeper PK conceded Opponent keeper PK saved Opponent keeper PK missed
24-25                   9                     9                     1.0                  1                    2                    0.5                         2                            1                         1                          0                        9                           9                        0                         0
25-26                   1                     2                     0.5                  3                    4                   0.75                    

## Season comparison findings

Both seasons contain 38 Premier League fixtures. All 14 independently reloaded `.xls` tables match the notebook dataframes. The paired `for` and `against` misc tables align by date and opponent across every fixture: `for` values are Liverpool's stats, and `against` values are the opponents' stats for those same fixtures. The paired comparison was independently recomputed from the workbooks and matched the dataframe results. Shooting-table footer rows were excluded by retaining match-date rows.

- **Results:** Liverpool fell from 84 points (25 wins, 9 draws, 4 losses) in 2024-25 to 60 points (17 wins, 9 draws, 12 losses) in 2025-26. Goal difference dropped from +45 to +10.
- **Chance creation:** Goals fell from 86 to 63. Shots declined from 647 to 588 (-9%), but shots on target fell more sharply from 231 to 175 (-24%). Shot-on-target share decreased from 35.7% to 29.8%; shot conversion from the shooting tables decreased from 13.1% to 10.4%.
- **Penalty kicks:** Shooting-table PK/PKatt show Liverpool scoring 9/9 in 2024-25 and 1/2 in 2025-26; opponents scored 1/2 and 3/4. These attempts and goals match the opposite keeper's PKatt/PKA row by row. Liverpool's keeper faced 2 then 4 penalties, conceding 1 and 3, saving 1 in each season. Keeper PKsv and PKm account for the remaining outcomes. Shooting and keeping penalty columns have no missing match values; blank PKwon/PKcon misc fields are excluded from the analysis.
- **Defensive activity:** Opponents' shots rose from 387 to 435 and goals conceded from 41 to 53. Liverpool's tackles won decreased from 399 (10.50 per match) to 307 (8.08), while interceptions decreased from 293 (7.71 per match) to 250 (6.58). Opponents' interceptions increased from 310 (8.16 per match) to 345 (9.08). These counts are consistent with Liverpool winning fewer defensive actions, but are not possession-adjusted and cannot establish why.
- **Attacking activity:** Liverpool's crosses increased from 725 (19.08 per match) to 762 (20.05), and offsides from 60 (1.58 per match) to 76 (2.00), despite lower shot totals. In 2025-26, Liverpool averaged 23.29 crosses per dropped-points match versus 16.06 in wins; tackles won averaged 7.24 versus 9.12. The same direction appears in 2024-25 for tackles won (7.92 versus 11.84), but crosses were similar across result groups (19.92 versus 18.64). These are descriptive patterns and may reflect game state or opponent differences, not causes.
- **Discipline and possession:** Liverpool fouls committed decreased from 11.32 to 10.11 per match and yellow cards from 1.76 to 1.50, so rising Liverpool fouls/cards do not explain the poorer season. Average possession rose from 58.1% to 59.3%; clean sheets fell from 14 to 10.
- **Offside imputation:** Opponent offsides are missing in one fixture per season and treated as zero; Liverpool's offside rows are complete. The comparison cell prints these replacement counts.

The workbooks contain no xG. Overall, the strongest measurable changes are lower shot accuracy and fewer recorded Liverpool tackles/interceptions alongside more opposition shots and goals; these are season-level patterns, not causal explanations.